In [2]:
from eventflow.data import load_train, make_train_examples
from eventflow.validation import add_folds, split
from eventflow.config import EVENT_CLASSES, TUNING_FOLD
from eventflow.features import make_features, CATEGORICAL

train = load_train()
examples = add_folds(make_train_examples(train))
X = make_features(train)
print("feature table:", X.shape)        # expect (1203963, 49)
X.iloc[95:100, :]

feature table: (1203963, 49)


,ev_lag0,sev_lag0,ev_lag1,sev_lag1,ev_lag2,sev_lag2,run_length,cnt5_COMPONENT_REPLACED,cnt5_CONTROL_ALERT,cnt5_COOLING_ACTIVATED,...,cnt30_EMERGENCY_STOP,cnt30_LOAD_DECREASE,cnt30_LOAD_INCREASE,cnt30_MAINTENANCE_CHECK,cnt30_PRESSURE_WARNING,cnt30_RECOVERY,cnt30_ROUTINE_CHECK,cnt30_SENSOR_RESET,cnt30_TEMP_WARNING,cnt30_VIBRATION_WARNING
95,7,0.0,3,2.0,2,0.0,1,0,0,1,...,1,0,2,5,0,3,0,0,6,8
96,2,0.0,7,0.0,3,2.0,1,0,0,2,...,1,0,1,5,0,3,0,0,6,8
97,2,0.0,2,0.0,7,0.0,2,0,0,3,...,1,0,1,4,0,3,0,0,6,8
98,9,1.0,2,0.0,2,0.0,1,0,0,2,...,1,0,1,4,0,3,0,0,6,8
99,12,1.0,9,1.0,2,0.0,1,0,0,2,...,1,0,1,3,0,3,0,0,7,8


In [3]:
row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()

by_hand = {
    "ev_lag0": EVENT_CLASSES.index(history[-1]),
    "ev_lag2": EVENT_CLASSES.index(history[-3]),
    "cnt30_TEMP_WARNING": history[-30:].count("TEMP_WARNING"),
    "cnt5_RECOVERY": history[-5:].count("RECOVERY"),
}
run = 1
while run < len(history) and history[-1 - run] == history[-1]:
    run += 1
by_hand["run_length"] = run

for name, value in by_hand.items():
    assert X.loc[row, name] == value, (name, X.loc[row, name], value)
print("Hand check passed:", by_hand)

Hand check passed: {'ev_lag0': 2, 'ev_lag2': 12, 'cnt30_TEMP_WARNING': 1, 'cnt5_RECOVERY': 2, 'run_length': 2}


In [ ]:
# NO-FUTURE TEST (leakage check)
# e.g. a diary of 250 events, cut at row 125:
#   full = make_features(rows 0..249) → features at row 125
#   cut  = make_features(rows 0..125) → features at row 125
#   same      → no feature used rows 126..249 → no leakage 
#   different → some feature peeked at the future → leakage 
lengths = train.groupby("sequence_id").size()
seq_id = lengths.idxmax()                                  # the longest machine
seq = train[train["sequence_id"] == seq_id].reset_index(drop=True)
full = make_features(seq)                                  # the whole diary

for c in [0, 10, 89, len(seq) // 2, len(seq) - 2]:         # several cut points
    cut = make_features(seq.iloc[:c + 1])                  # the diary cut after row c
    assert full.iloc[c].equals(cut.iloc[c]), c
print(f"No-future test passed on {seq_id} ({len(seq)} events): features at a cut do not change when later events are removed")

No-future test passed on TR00160 (250 events): features at a cut do not change when later events are removed
